## Carga de documentos 

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader1 = PyPDFLoader("../datapdf/programacion_datos.pdf")
loader2 = PyPDFLoader("../datapdf/programacion_python.pdf")
loader3 = PyPDFLoader("../datapdf/arquitectura_ti.pdf")
loader4 = PyPDFLoader("../datapdf/ingenieria_datos.pdf")

documents = loader1.load() + loader2.load() + loader3.load() + loader4.load()

print(f"Páginas cargadas: {len(documents)}")

incorrect startxref pointer(1)
parsing for Object Streams


Páginas cargadas: 35


## Chunking 

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # cuántos caracteres por fragmento. Más grande = más contexto, pero más costoso de procesar.
    chunk_overlap=100,   # cuántos caracteres se repiten entre chunks. Evita perder información en los bordes.
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Total de chunks: {len(chunks)}")
print(f"Promedio de caracteres por chunk: {sum(len(c.page_content) for c in chunks)//len(chunks)}")
print(f"\nPrimer chunk:\n{chunks[0].page_content}")

Total de chunks: 79
Promedio de caracteres por chunk: 415

Primer chunk:
Sílabo
170332 - Programación para la Ciencia de Datos
I. Información general
Nombre del Curso: Programación para la Ciencia de Datos
Código del curso: 170332
Departamento Académico: Ingeniería
Créditos: 4
Horas Teoría: 3
Horas Práctica: 2
Periodo Académico: 2023-01-PRE
Sección: A
Modalidad: Presencial
Idioma: Español
Docente: 
Email docente: u.rojasv@up.edu.pe
II. Introducción
El presente curso brindará a los estudiantes una visión a un nivel básico e intermedio de conceptos


## Embedding 

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"
)

print("Modelo de embeddings cargado")

C:\Users\emerd\AppData\Local\Temp\ipykernel_4864\2552464410.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo de embeddings cargado


## Base Vectorial

In [4]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("Base vectorial creada")

Base vectorial creada


## LLM Definition

In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

google_key = os.getenv("GOOGLE_API_KEY")

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
      model="gemini-2.5-flash-lite", # gemini-2.5-flash-lite / gemini-2.5-flash
      google_api_key=google_key,
      temperature=0.4,
      max_output_tokens=1024
  )

## Memoria Conversacional

In [7]:
from langchain_core.messages import HumanMessage, AIMessage

class MemoriaConversacional:
    """
    Memoria manual para control del historial
    """

    def __init__(self, max_mensajes=10):
        self.historial = []  # Lista de mensajes
        self.max_mensajes = max_mensajes

    def agregar(self, mensaje_usuario, respuesta_ia):
        """Agrega un intercambio al historial"""
        self.historial.append(HumanMessage(content=mensaje_usuario))
        self.historial.append(AIMessage(content=respuesta_ia))

        # Limitar tamaño del historial
        if len(self.historial) > self.max_mensajes * 2:
            self.historial = self.historial[-self.max_mensajes * 2:]

    def obtener_historial_formateado(self):
        """Devuelve historial como string para el prompt"""
        if not self.historial:
            return "No hay conversación previa."

        lines = []
        for msg in self.historial:
            if isinstance(msg, HumanMessage):
                lines.append(f"Usuario: {msg.content}")
            else:
                lines.append(f"Asistente: {msg.content}")
        return "\n".join(lines)

    def limpiar(self):
        """Reinicia la memoria"""
        self.historial = []
        print("🔄 Memoria reiniciada")

    def ver_historial(self):
        """Muestra el historial formateado"""
        print("\n📜 HISTORIAL DE CONVERSACIÓN:")
        print("=" * 60)
        for msg in self.historial:
            if isinstance(msg, HumanMessage):
                print(f"👤 Usuario: {msg.content}")
            else:
                print(f"🤖 Asistente: {msg.content}")
                print("-" * 40)
        print("=" * 60)

print("✅ Clase MemoriaConversacional creada")

✅ Clase MemoriaConversacional creada


##  Prompt

In [ ]:
from prompts.conversational_prompt import PROMPT_CONVERSACIONAL
from prompts.rag_prompt import PROMPT_CONVERSACIONAL

In [9]:
# Guardrail Decorator
PALABRAS_PROHIBIDAS = {
    "chiste", "broma", "joke", "chismes", "gossip", "rumor", "cotilleo",
    "anécdota", "historia", "cuento", "diversión", "entretenimiento",
    "noticia", "escándalo", "política", "deporte", "celebridad", "famoso",
    "música", "película", "serie", "juego", "videojuego", "hobby", "pasatiempo"
}

def check(func):
    def wrapper(self, pregunta, *args, **kwargs):
        p = pregunta.lower()
        for palabra in PALABRAS_PROHIBIDAS:
            if palabra in p:
                return "Lo siento, no puedo responder a preguntas sobre chistes, chismes o temas no relacionados con el aprendizaje académico. ¿En qué puedo ayudarte con temas del curso?"
        return func(self, pregunta, *args, **kwargs)
    return wrapper

print("✅ Decorador de guardrail creado")

✅ Decorador de guardrail creado


## Intent Classifier

In [10]:
class ClasificadorIntencion:
    """
    Clasifica la intención del usuario sin usar LLM (rápido y gratuito)
    """

    def __init__(self):
        self.saludos = {
            "hola", "buenos días", "buenas tardes", "buenas noches",
            "que tal", "cómo va", "saludos"
        }

        self.despedidas = {
            "adiós", "chao", "nos vemos", "hasta luego", "hasta pronto",
            "bye", "gracias", "muchas gracias", "te lo agradezco"
        }

        self.conversacional = {
            "cómo estás", "cómo te va", "qué tal", "todo bien",
            "qué eres", "eres un bot",
            "cómo te llamas", "qué sabes hacer", "para qué sirves"
        }

        self.academico = {
            "curso", "sesión", "clase", "estadística", "enseña", "aprender",
            "documento", "tema", "contenido", "material", "estudiar",
            "ejercicio", "práctica", "teoría", "concepto", "definición"
        }

    def clasificar(self, pregunta):
        """Devuelve: 'RAG' o 'CONVERSACIONAL'"""
        p = pregunta.lower().strip()

        # Prioridad: primero revisar palabras clave
        for palabra in self.saludos:
            if palabra in p:
                return "CONVERSACIONAL"

        for palabra in self.despedidas:
            if palabra in p:
                return "CONVERSACIONAL"

        for palabra in self.conversacional:
            if palabra in p:
                return "CONVERSACIONAL"

        for palabra in self.academico:
            if palabra in p:
                return "RAG"

        # Si la pregunta es muy corta (menos de 5 palabras), asumir conversacional
        if len(p.split()) < 5:
            return "CONVERSACIONAL"

        # Por defecto, usar RAG
        return "RAG"

    def ver_categorias(self):
        """Muestra las palabras clave configuradas (para debug)"""
        print("📋 CATEGORÍAS DEL CLASIFICADOR:")
        print(f"   Saludos: {len(self.saludos)} palabras")
        print(f"   Despedidas: {len(self.despedidas)} palabras")
        print(f"   Conversacional: {len(self.conversacional)} palabras")
        print(f"   Académico: {len(self.academico)} palabras")

# Instanciar clasificador (una sola vez)
clasificador = ClasificadorIntencion()
print("✅ Clasificador de intención listo")

✅ Clasificador de intención listo


## Chatbot Academico con RAG + Memoria

In [11]:
class ChatbotAcademico:
    def __init__(self, vectorstore, llm, clasificador, umbral=1.3):  # ← añade clasificador
        self.vectorstore = vectorstore
        self.llm = llm
        self.clasificador = clasificador  # ← nuevo
        self.umbral = umbral
        self.memoria = []
        self.num_interacciones = 0

    @check
    def chat(self, pregunta):
        self.num_interacciones += 1
        memoria_reciente = "\n".join(self.memoria[-10:])

        # Clasificar
        if self.clasificador.clasificar(pregunta) == "CONVERSACIONAL":
            # Ruta conversacional
            prompt = PROMPT_CONVERSACIONAL.format(
                historial=memoria_reciente,
                pregunta=pregunta
            )
        else:
            # Ruta RAG
            resultados = self.vectorstore.similarity_search_with_score(pregunta, k=3)
            docs_relevantes = [doc for doc, score in resultados if score < self.umbral]
            texto_contexto = "\n".join([doc.page_content for doc in docs_relevantes]) if docs_relevantes else "No se encontró información relevante."

            prompt = PROMPT_RAG.format(
                contexto=texto_contexto,
                historial=memoria_reciente,
                pregunta=pregunta
            )

        respuesta = self.llm.invoke(prompt).content
        self.memoria.append(f"Usuario: {pregunta}")
        self.memoria.append(f"Asistente: {respuesta}")
        return respuesta

    def reset(self):
        self.memoria = []
        self.num_interacciones = 0

    def ver_memoria(self):
        print("\n".join(self.memoria))

    def stats(self):
        print(f"Interacciones: {self.num_interacciones}")
        print(f"Mensajes: {len(self.memoria)}")

In [12]:
# Crear instancia del chatbot
chatbot = ChatbotAcademico(vectorstore=vectorstore, llm=llm,clasificador=clasificador)

def preguntar(pregunta):
    respuesta = chatbot.chat(pregunta)
    print(f"🤖 {respuesta}\n")

In [13]:
preguntar("Quiero aprender SQL, qué debería estudiar?")

🤖 Para aprender SQL, deberías estudiar:

*   Estructura básica del SQL
*   Sentencias DDL
*   Sentencias DML
*   Sentencias DCL
*   Copias de Seguridad y Recuperación de información
*   Unidad de Aprendizaje 5: Automatización de procesos en T-SQL, que incluye:
    *   Diferencias entre bases de datos SQL vs NoSQL
    *   Big Data
    *   Introducción al PL/SQL
    *   Creación y manejo de procedimientos, funciones y disparadores

